# Lab 06 — Morphological Segmentation & Watershed Separation

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Biomedical histological segmentation of an ovarian follicle (`follicule.bmp`):
   - **Antrum** extraction via blue-channel thresholding, connected-component selection, and hole filling
   - **Theca** ring extraction via morphological dilation and set difference
   - **Vascularization** detection inside the theca layer
2. Separation of touching objects (`circles.TIF`) using Euclidean Distance Transform, morphological regional maxima, and the Watershed transform

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import imageio.v3 as imageio
from skimage import morphology, measure, segmentation

IMAGES_DIR = Path("../Images")

## 1. Ovarian Follicle Segmentation

We isolate three anatomical structures from `follicule.bmp` using channel thresholding and mathematical morphology.

In [ ]:
follicle = imageio.imread(IMAGES_DIR / "follicule.bmp")
blue_channel = follicle[:, :, 2]

# 1. Antrum: bright region in blue channel containing seed (300, 300), then fill holes
antrum_mask = blue_channel > 220
labeled = measure.label(antrum_mask, connectivity=2)
antrum = labeled == labeled[300, 300]
antrum = ndi.binary_fill_holes(antrum)

# 2. Theca: ring around the antrum via dilation (disk radius 40) minus antrum
se40 = morphology.disk(40)
theca = morphology.dilation(antrum, footprint=se40) & ~antrum

# 3. Vascularization: dark vessels (B < 140) located within the theca
vascularization = (blue_channel < 140) & theca

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
panels = [
    (follicle, "Original Histology", None),
    (antrum, "Antrum", "gray"),
    (theca, "Theca Ring (r=40)", "gray"),
    (vascularization, "Vascularization", "gray"),
]
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. Separating Touching Objects via Distance Transform & Watershed

To separate overlapping circular objects in `circles.TIF`:
1. Compute the Euclidean Distance Transform (EDT)
2. Extract regional maxima via morphological reconstruction (`rmax`) to serve as watershed markers
3. Run the Watershed transform on the inverted distance map (`-dm`)

In [ ]:
def regional_maxima(image: np.ndarray) -> np.ndarray:
    """Compute regional maxima via morphological geodesic reconstruction."""
    scaled = image.astype(np.float64)
    scaled = (scaled / np.max(scaled) * (2**31 - 2)).astype(np.int32)
    h = 1
    reconstructed = morphology.reconstruction(scaled, scaled + h, method="dilation")
    return (scaled + h - reconstructed) > 0


circles = imageio.imread(IMAGES_DIR / "circles.TIF") > 0
distance_map = ndi.distance_transform_edt(circles)
local_maxima = regional_maxima(distance_map)

markers, num_markers = ndi.label(local_maxima, np.ones((3, 3)))
watershed_labels = segmentation.watershed(-distance_map, markers, watershed_line=True)

# Apply watershed separating lines to original binary mask
separated_mask = circles & (watershed_labels > 0)
separated_objects, num_objects = ndi.label(separated_mask, np.ones((3, 3)))
print(f"Detected markers: {num_markers} | Separated objects: {num_objects}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(circles, cmap="gray")
axes[0].set_title("Original Binary Circles")

axes[1].imshow(distance_map, cmap="magma")
axes[1].set_title("Euclidean Distance Map")

axes[2].imshow(watershed_labels == 0, cmap="gray")
axes[2].set_title("Watershed Dividing Lines")

axes[3].imshow(separated_objects, cmap="nipy_spectral")
axes[3].set_title(f"Separated Labels ({num_objects})")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()